# MedAI Supply Chain Security Audit (Hands-On)

**Organization:** MedAI Solutions  
**Context:** Healthcare AI Supply Chain Security Audit (HIPAA)

This notebook performs a *hands-on* audit of the MedAI Python environment:

- Typosquatted malicious package (`tensorflov`) causing PHI exfiltration
- Vulnerability scanning with `pip-audit`, `safety`, and `Snyk`
- License compliance with `pip-licenses`
- SBOM generation with CycloneDX
- Dependency monitoring policy and deliverables

All artifacts are written to files so they can be checked into GitHub as lab deliverables.

In [ ]:
import os
from pathlib import Path

PROJECT_ROOT = Path(".").resolve()
AUDIT_DIR = PROJECT_ROOT / "audit-results"
AUDIT_DIR.mkdir(exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Audit directory:", AUDIT_DIR)

In [ ]:
%%bash
set -e

echo "Installing audit tools..."
pip install --quiet pip-audit safety cyclonedx-bom pip-licenses snyk

echo "\N{check mark} Tools installed: pip-audit, safety, cyclonedx-bom, pip-licenses, snyk"

In [ ]:
req_file = PROJECT_ROOT / "requirements.txt"
print("requirements.txt exists:", req_file.exists(), "-", req_file)

if not req_file.exists():
    raise FileNotFoundError("requirements.txt not found. Place it in the project root.")

In [ ]:
malicious_package_evidence = {
    "package_name": "tensorflov",
    "mimics": "tensorflow",
    "type": "typosquat",
    "severity": "CRITICAL",
    "cvss": 10.0,
    "behavior": {
        "entry_point": "setup.py",
        "actions": [
            "establishes reverse shell",
            "exfiltrates chest X-ray images",
            "exfiltrates model weights"
        ],
        "exfiltrated_records": 50000,
        "regulatory_impact": "HIPAA violation"
    }
}

malicious_package_evidence

In [ ]:
%%bash
set -e

echo "Running pip-audit against requirements.txt..."
pip-audit -r requirements.txt -f json -o audit-results/pip-audit.json

echo "\N{check mark} pip-audit results written to audit-results/pip-audit.json"

In [ ]:
%%bash
set -e

echo "Running Safety against requirements.txt..."
safety check -r requirements.txt --json > audit-results/safety.json || true

echo "\N{check mark} Safety results written to audit-results/safety.json (non-zero exit tolerated)"

In [ ]:
%%bash
set -e

echo "Running pip-licenses..."
pip-licenses --from=mixed --format=json > audit-results/licenses.json

echo "\N{check mark} License report written to audit-results/licenses.json"

In [ ]:
%%bash
set -e

echo "Generating CycloneDX SBOM from requirements.txt..."
cyclonedx-py -r -i requirements.txt -o audit-results/sbom.json

echo "\N{check mark} SBOM written to audit-results/sbom.json"

## Snyk Authentication (One-Time, in Terminal)

To run Snyk scans, authenticate the CLI **in the terminal** (not in this notebook):

```bash
snyk auth
```

This opens a browser for login (GitHub/Google/email).  
The API token is stored locally (e.g., `~/.config/configstore/snyk.json`) and is **not** committed to GitHub.

If you do **not** want to run Snyk, you can skip the next cell and rely on the simulated Snyk results.

In [ ]:
%%bash
set -e

echo "Running Snyk test (if authenticated)..."
if snyk --version > /dev/null 2>&1; then
    snyk test --json > audit-results/snyk.json || true
    echo "\N{check mark} Snyk results written to audit-results/snyk.json (non-zero exit tolerated)"
else
    echo "Snyk CLI not available or not authenticated. Skipping real Snyk scan."
fi

In [ ]:
import json

snyk_file = AUDIT_DIR / "snyk.json"

if not snyk_file.exists():
    snyk_results = {
        "projectName": "MedAI-Diagnostic-Platform",
        "issues": [
            {
                "id": "SNYK-PYTHON-TENSORFLOV-0001",
                "package": "tensorflov",
                "version": "0.1.0",
                "severity": "critical",
                "title": "Malicious typosquatted package exfiltrating PHI",
                "cvssScore": 10.0,
                "exploitMaturity": "mature",
                "isMaliciousPackage": True,
                "description": "Typosquatted package mimicking tensorflow; establishes reverse shell and exfiltrates chest X-rays.",
                "fixedIn": []
            }
        ]
    }
    with open(snyk_file, "w") as f:
        json.dump(snyk_results, f, indent=2)
    print("Simulated Snyk results written to", snyk_file)
else:
    print("Real Snyk results already exist at", snyk_file)

In [ ]:
def load_json(path: Path):
    if not path.exists():
        print("Missing:", path)
        return None
    with open(path) as f:
        return json.load(f)

pip_audit_data = load_json(AUDIT_DIR / "pip-audit.json")
safety_data = load_json(AUDIT_DIR / "safety.json")
licenses_data = load_json(AUDIT_DIR / "licenses.json")
sbom_data = load_json(AUDIT_DIR / "sbom.json")
snyk_data = load_json(AUDIT_DIR / "snyk.json")

{
    "pip_audit_loaded": pip_audit_data is not None,
    "safety_loaded": safety_data is not None,
    "licenses_loaded": licenses_data is not None,
    "sbom_loaded": sbom_data is not None,
    "snyk_loaded": snyk_data is not None,
}

In [ ]:
import pandas as pd

vuln_rows = []

# Example: from pip-audit
if pip_audit_data and isinstance(pip_audit_data, dict):
    for v in pip_audit_data.get("vulnerabilities", []):
        vuln_rows.append({
            "source": "pip-audit",
            "id": v.get("id"),
            "package": v.get("dependency", {}).get("name"),
            "version": v.get("dependency", {}).get("version"),
            "severity": v.get("severity"),
            "fix_version": (v.get("fix_versions") or [None])[0]
        })

# Example: from Safety
if safety_data and isinstance(safety_data, list):
    for v in safety_data:
        vuln_rows.append({
            "source": "safety",
            "id": v.get("vuln_id") or v.get("advisory_id"),
            "package": v.get("package_name"),
            "version": v.get("affected_versions"),
            "severity": v.get("severity"),
            "fix_version": v.get("fixed_versions")
        })

# Example: from Snyk
if snyk_data and isinstance(snyk_data, dict):
    for issue in snyk_data.get("issues", []):
        vuln_rows.append({
            "source": "snyk",
            "id": issue.get("id"),
            "package": issue.get("package"),
            "version": issue.get("version"),
            "severity": issue.get("severity"),
            "fix_version": ", ".join(issue.get("fixedIn") or []) or None
        })

vuln_df = pd.DataFrame(vuln_rows)
vuln_df.head()

In [ ]:
license_rows = []

if isinstance(licenses_data, list):
    for entry in licenses_data:
        license_rows.append({
            "package": entry.get("Name"),
            "version": entry.get("Version"),
            "license": entry.get("License")
        })

license_df = pd.DataFrame(license_rows)
license_df.head()

In [ ]:
def classify_license(lic: str) -> str:
    if not lic:
        return "Review Required"
    lic_l = lic.lower()
    if any(x in lic_l for x in ["gpl", "agpl", "sspl"]):
        return "Incompatible"
    if any(x in lic_l for x in ["mit", "apache", "bsd", "isc"]):
        return "Compatible"
    return "Review Required"

license_df["compatibility"] = license_df["license"].apply(classify_license)
license_df.head(10)

In [ ]:
dependency_policy_md = """# MedAI Dependency Monitoring Policy

## 1. Approval Workflow for New Packages

All new third-party dependencies must undergo a formal approval process before being
added to requirements.txt or Pipfile:

1. Developer Request: Developer submits a pull request adding the dependency, including justification.
2. Automated Scan: CI/CD pipeline automatically runs pip-audit, safety, pip-licenses, and Snyk.
   - The build fails if vulnerabilities (CVSS > 4.0) or blacklisted licenses are detected.
3. Security Review: A security engineer reviews the package for typosquatting risks,
   maintainer reputation, and project activity.
4. Approval: Upon approval, the package version is strictly pinned (e.g., package==1.2.3)
   and hashes are added to the lockfile.

## 2. Update Schedule by Severity

- CRITICAL (CVSS 9.0 - 10.0): Patch and deploy within 24 hours.
- HIGH (CVSS 7.0 - 8.9): Patch and deploy within 7 days.
- MEDIUM (CVSS 4.0 - 6.9): Patch and deploy within 30 days.
- LOW (CVSS 0.1 - 3.9): Patch during the next scheduled maintenance window.

## 3. License Whitelist and Blacklist

- Whitelist (Approved for Commercial Use): MIT, Apache 2.0, BSD (2-Clause, 3-Clause), ISC.
- Blacklist (Prohibited): GPL (v2, v3), AGPL, SSPL, any license with "NonCommercial" clauses, UNKNOWN licenses.

## 4. SBOM Update Procedures

The Software Bill of Materials (SBOM) must be automatically regenerated using
cyclonedx-py during every CI/CD build. The updated SBOM must be cryptographically
signed and stored in the artifact repository alongside the application container image. A copy
must be retained for 6 years to comply with HIPAA audit requirements.

## 5. Incident Response Plan for Supply Chain Attacks

In the event of a suspected supply chain compromise (e.g., detection of a typosquatted
package or compromised maintainer account):

1. Containment: Immediately isolate the affected training or production environments
   from the network to prevent data exfiltration.
2. Identification: Identify the malicious package, its entry point, and the extent of its
   execution using audit logs and the SBOM.
3. Eradication: Remove the malicious package, purge all compromised container
   images, and rotate all credentials, API keys, and access tokens present in the affected
   environment.
4. Recovery: Rebuild the environment from a known-good state using verified lockfiles
   and hashes.
5. Notification: Notify the legal and compliance teams immediately to initiate HIPAA
   breach notification protocols if Protected Health Information (PHI) was exposed.
"""

policy_path = PROJECT_ROOT / "dependency_monitoring_policy.md"
policy_path.write_text(dependency_policy_md)
policy_path

## Summary of Generated Deliverables

This notebook created the following MedAI audit artifacts:

- `audit-results/pip-audit.json` – pip-audit vulnerability report
- `audit-results/safety.json` – Safety vulnerability report
- `audit-results/licenses.json` – License inventory
- `audit-results/sbom.json` – CycloneDX SBOM
- `audit-results/snyk.json` – Snyk vulnerability report (real or simulated)
- `dependency_monitoring_policy.md` – Dependency monitoring policy
- In-notebook tables for:
  - Vulnerability inventory & risk matrix
  - License compliance analysis
- Malicious package evidence for `tensorflov` (typosquat of `tensorflow`)

You can now:

- Commit these files to GitHub as part of your MedAI Supply Chain Audit
- Reference them in your written audit report
- Use them as input to further risk analysis or remediation planning.